# Notebook 10: Cold Start Adaptation

## Objective

Handle new users and devices with no historical behaviour by using
similar entity behaviour profiles instead of treating unknown entities
as malicious.

## Workflow

Existing Behaviour Profiles
        ↓
Peer Group Generation
        ↓
New Entity Detection
        ↓
Similarity Matching
        ↓
Risk Estimation
        ↓
Cold Start Report

In [1]:
import warnings
warnings.filterwarnings("ignore")

import os
import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
os.makedirs("reports/cold_start",exist_ok=True)

os.makedirs("models/cold_start",exist_ok=True)

print("Directories created")

Directories created


In [3]:
features = pd.read_csv("/content/deeptrace_features_FINAL.csv")


print(features.shape)

features.head()

(239480, 49)


,department,role,office_location,device,network_type,authentication,event_type,application,resource,risk_baseline,...,upload_count,database_query_count,file_access_count,admin_action_count,powershell_execution_count,is_privileged_user,is_admin,remote_access,high_risk_event,foreign_login
0,1.0,0.45,0.8,0.25,0.0,1.000000,0.500000,0.03125,0.045455,0.5,...,0.0,0.1875,0.0,0.230769,0.0,0.0,0.0,0.0,0.0,0.0
1,1.0,0.45,0.8,0.25,0.0,0.333333,0.000000,0.00000,0.409091,0.5,...,0.0,0.1875,0.0,0.230769,0.0,0.0,0.0,0.0,1.0,0.0
2,1.0,0.45,0.8,0.25,0.0,1.000000,0.083333,0.90625,0.636364,0.5,...,0.0,0.1875,0.0,0.230769,0.0,0.0,0.0,0.0,0.0,0.0
3,1.0,0.45,0.8,0.25,0.0,0.000000,0.083333,0.21875,0.090909,0.5,...,0.0,0.1875,0.0,0.230769,0.0,0.0,0.0,0.0,0.0,0.0
4,1.0,0.45,0.8,0.25,0.0,0.666667,0.083333,0.37500,0.636364,0.5,...,0.0,0.1875,0.0,0.230769,0.0,0.0,0.0,0.0,0.0,0.0


In [4]:
profile_data = features.sample(5000,random_state=42)

profile_data.shape

(5000, 49)

In [5]:
scaler = StandardScaler()

scaled_profiles = scaler.fit_transform(profile_data)

print(scaled_profiles.shape)

(5000, 49)


In [6]:
kmeans = KMeans(
    n_clusters=5,
    random_state=42,
    n_init=10
)


clusters = kmeans.fit_predict(
    scaled_profiles
)


profile_data["cluster"] = clusters


profile_data.head()

,department,role,office_location,device,network_type,authentication,event_type,application,resource,risk_baseline,...,database_query_count,file_access_count,admin_action_count,powershell_execution_count,is_privileged_user,is_admin,remote_access,high_risk_event,foreign_login,cluster
125817,0.000000,0.90,0.2,0.25,0.0,1.000000,0.083333,0.12500,0.090909,0.5,...,0.125,0.444444,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,3
19429,1.000000,0.65,0.0,0.25,1.0,0.333333,1.000000,0.96875,0.954545,0.5,...,0.250,0.000000,0.153846,0.000000,0.0,0.0,1.0,0.0,0.0,0
141984,0.428571,0.30,0.6,1.00,0.0,1.000000,0.333333,0.59375,0.227273,0.5,...,0.000,0.333333,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,1
197245,1.000000,0.65,0.2,1.00,1.0,0.333333,1.000000,0.96875,0.954545,0.5,...,0.250,0.111111,0.307692,0.000000,0.0,0.0,1.0,0.0,0.0,0
14708,0.571429,0.60,1.0,1.00,0.0,0.000000,0.416667,0.84375,0.181818,0.0,...,0.000,0.222222,0.846154,0.428571,0.0,0.0,0.0,0.0,0.0,4


In [7]:
import joblib


joblib.dump(scaler,"models/cold_start/profile_scaler.pkl")


joblib.dump(kmeans,"models/cold_start/behaviour_cluster_model.pkl")


print("Cold start models saved")

Cold start models saved


In [8]:
new_entity = features.sample(1,random_state=10)


new_entity

,department,role,office_location,device,network_type,authentication,event_type,application,resource,risk_baseline,...,upload_count,database_query_count,file_access_count,admin_action_count,powershell_execution_count,is_privileged_user,is_admin,remote_access,high_risk_event,foreign_login
50168,0.285714,0.2,0.2,0.0,0.0,1.0,0.333333,0.59375,0.227273,1.0,...,0.0,0.3125,0.5,0.0,0.0,1.0,1.0,0.0,0.0,0.0


In [9]:
new_scaled = scaler.transform(
    new_entity
)


similarity = cosine_similarity(
    new_scaled,
    scaled_profiles
)


similar_entity_index = np.argmax(
    similarity
)


similarity_score = similarity[0][similar_entity_index]


print(
    "Similarity:",
    similarity_score
)

Similarity: 0.8136328458348294


In [10]:
cold_start_score = (
    1 - similarity_score
) * 100


if cold_start_score < 25:
    risk = "Low"

elif cold_start_score < 50:
    risk = "Medium"

elif cold_start_score < 75:
    risk = "High"

else:
    risk = "Critical"


print(
    "Cold Start Risk:",
    round(cold_start_score,2)
)

print(
    "Risk Level:",
    risk
)

Cold Start Risk: 18.64
Risk Level: Low


In [11]:
cold_start_report = pd.DataFrame({

    "Entity_Type":["New User"],
    "Similarity_Score":[similarity_score],
    "Cold_Start_Risk":[cold_start_score],
    "Risk_Level":[risk]

})


cold_start_report

,Entity_Type,Similarity_Score,Cold_Start_Risk,Risk_Level
0,New User,0.813633,18.636715,Low


In [12]:
cold_start_report.to_csv(
    "reports/cold_start/cold_start_predictions.csv",
    index=False
)


print("Cold start report saved")

Cold start report saved


In [13]:
summary = f"""

DeepTrace Cold Start Adaptation Report

Purpose:
Score new users/devices without historical behaviour.

Approach:
Peer behaviour similarity matching.

Similarity Score:
{similarity_score:.4f}

Cold Start Risk:
{cold_start_score:.2f}

Risk Level:
{risk}

The system avoids automatically marking unknown entities
as malicious and uses similar behavioural profiles.
"""


with open(
    "reports/cold_start/cold_start_analysis.txt",
    "w"
) as f:
    f.write(summary)


print(summary)



DeepTrace Cold Start Adaptation Report

Purpose:
Score new users/devices without historical behaviour.

Approach:
Peer behaviour similarity matching.

Similarity Score:
0.8136

Cold Start Risk:
18.64

Risk Level:
Low

The system avoids automatically marking unknown entities
as malicious and uses similar behavioural profiles.



In [14]:
cold_start_results = []

sample_users = features.sample(
    100,
    random_state=42
)


for idx, row in sample_users.iterrows():

    entity = row.values.reshape(1,-1)

    entity_scaled = scaler.transform(entity)

    similarity = cosine_similarity(
        entity_scaled,
        scaled_profiles
    )[0].max()


    risk_score = (1-similarity)*100


    if risk_score < 25:
        level="Low"

    elif risk_score <50:
        level="Medium"

    elif risk_score <75:
        level="High"

    else:
        level="Critical"


    cold_start_results.append({

        "Entity_ID":idx,
        "Similarity":similarity,
        "Risk_Score":risk_score,
        "Risk_Level":level

    })


cold_start_df = pd.DataFrame(
    cold_start_results
)


cold_start_df.head()

,Entity_ID,Similarity,Risk_Score,Risk_Level
0,125817,1.0,1.110223e-14,Low
1,19429,1.0,2.220446e-14,Low
2,141984,1.0,0.000000e+00,Low
3,197245,1.0,-2.220446e-14,Low
4,14708,1.0,0.000000e+00,Low


In [15]:
cold_start_df["Risk_Level"].value_counts()

,count
Risk_Level,
Low,100


## Cold Start Stress Testing

To validate robustness, DeepTrace is tested against an unseen behaviour
profile that does not match existing user/device patterns.

The objective is to ensure unusual new entities receive higher risk
scores instead of being incorrectly accepted.

In [16]:
unknown_entity = np.random.uniform(
    0,
    1,
    size=(1,49)
)

unknown_entity.shape

(1, 49)

In [17]:
unknown_scaled = scaler.transform(
    unknown_entity
)


unknown_similarity = cosine_similarity(
    unknown_scaled,
    scaled_profiles
)[0].max()


unknown_risk = (
    1 - unknown_similarity
) * 100


if unknown_risk < 25:
    unknown_level = "Low"

elif unknown_risk < 50:
    unknown_level = "Medium"

elif unknown_risk < 75:
    unknown_level = "High"

else:
    unknown_level = "Critical"


print("Similarity:", round(unknown_similarity,4))
print("Risk Score:", round(unknown_risk,2))
print("Risk Level:", unknown_level)

Similarity: 0.3516
Risk Score: 64.84
Risk Level: High


In [21]:
comparison = pd.DataFrame({

    "Scenario":[
        "Normal New Entity",
        "Unknown Behaviour Entity"
    ],

    "Risk_Level":[
        cold_start_df["Risk_Level"].iloc[0],
        unknown_level
    ],

    "Risk_Score":[
        round(cold_start_df["Risk_Score"].iloc[0],2),
        round(unknown_risk,2)
    ]

})


comparison

,Scenario,Risk_Level,Risk_Score
0,Normal New Entity,Low,0.00
1,Unknown Behaviour Entity,High,64.84


In [22]:
comparison.to_csv(
    "reports/cold_start/cold_start_scenario_comparison.csv",
    index=False
)

print("Cold start comparison saved")

Cold start comparison saved


# Conclusion

DeepTrace handles cold-start scenarios by using behaviour similarity
matching instead of assuming unknown users or devices are malicious.

New entities are compared against existing behavioural profiles to
estimate risk without requiring historical activity.

Normal behaviour patterns receive low risk scores, while unseen
behaviour patterns receive elevated risk scores for further analysis.

This reduces false positives while maintaining detection capability
against unknown entities.